[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/run_ersilia_model_colab.ipynb)

# Run an Ersilia model in Colab

Pick a model from the [Ersilia Model Hub](https://catalog.ersilia.io), fetch it and get predictions.

In [ ]:
#@title Setup: run this once, then forget about it { display-mode: "form" }
#@markdown Installs Apptainer and defines `ErsiliaRunner`. Takes about a minute.
import hashlib
import io
import json
import os
import pathlib
import shutil
import subprocess
import time
import urllib.error
import urllib.request

import pandas as pd
from tqdm.auto import tqdm


def _sh(command, timeout=None):
    """Run a shell command and return the completed process."""
    return subprocess.run(
        command, shell=True, capture_output=True, text=True, timeout=timeout
    )


# Ersilia marks input it cannot read with this string, rather than dropping the row.
UNPROCESSABLE_INPUT = "UNPROCESSABLE_INPUT"


def compound_key(smiles):
    """Return Ersilia's key for a molecule: the MD5 checksum of its SMILES.

    This mirrors ``CompoundIdentifier.encode`` in ersilia-os/ersilia. The key is a
    checksum of the string itself, not an InChIKey, so it is deterministic but says
    nothing about the chemistry: two spellings of the same molecule give two keys.

    Parameters
    ----------
    smiles : str
        The SMILES string of the compound.

    Returns
    -------
    str
        A 32-character hexadecimal checksum, or ``"UNPROCESSABLE_INPUT"``.
    """
    if (
        not isinstance(smiles, str)
        or not smiles.strip()
        or smiles == UNPROCESSABLE_INPUT
    ):
        return UNPROCESSABLE_INPUT
    return hashlib.md5(smiles.encode()).hexdigest()


class ErsiliaRunner:
    """Download an Ersilia model image, serve it, and run molecules through it.

    The images live in a public S3 bucket, one file per model and version. Each one
    carries a small web server; predictions are made by posting to it.

    Parameters
    ----------
    model_id : str
        Identifier from the Ersilia Model Hub, for example ``"eos42ez"``.
    version : str
        Version of the image, for example ``"v1"``.
    port : int
        Local port to serve the model on.

    Examples
    --------
    >>> runner = ErsiliaRunner("eos42ez", "v1")
    >>> runner.run(["CCO", "c1ccccc1"])
    """

    BUCKET = "https://models-sif.s3.eu-north-1.amazonaws.com"

    def __init__(self, model_id, version="v1", port=8000):
        self.model_id = model_id
        self.version = version
        self.port = port
        self.image = f"/content/{model_id}_{version}.sif"
        self.log = pathlib.Path(f"/content/{model_id}_serve.log")
        self.serving = False

    def __repr__(self):
        state = "serving" if self.serving else "not started"
        return f"ErsiliaRunner({self.model_id!r}, {self.version!r}) [{state}]"

    @property
    def url(self):
        """Where this model's image lives."""
        return f"{self.BUCKET}/{self.model_id}_{self.version}.sif"

    def size(self):
        """Size of the image in bytes, or None if there is no such model or version."""
        try:
            request = urllib.request.Request(self.url, method="HEAD")
            with urllib.request.urlopen(request, timeout=30) as response:
                return int(response.headers["Content-Length"])
        except urllib.error.HTTPError:
            # The bucket forbids listing, so a missing object answers 403, not 404.
            return None

    def fetch(self):
        """Download the image, unless it is already here in full."""
        expected = self.size()
        if expected is None:
            raise ValueError(
                f"No image for {self.model_id} {self.version}. Looked for {self.url}. "
                "Check the identifier and the version."
            )

        if os.path.exists(self.image) and os.path.getsize(self.image) == expected:
            print(f"{self.model_id} {self.version} is already downloaded.")
            return self

        with urllib.request.urlopen(self.url, timeout=60) as response:
            with open(self.image, "wb") as handle:
                with tqdm(
                    total=expected,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"{self.model_id} {self.version}",
                ) as bar:
                    while True:
                        chunk = response.read(1 << 20)
                        if not chunk:
                            break
                        handle.write(chunk)
                        bar.update(len(chunk))

        if os.path.getsize(self.image) != expected:
            raise IOError("The download is incomplete. Call fetch() again.")
        return self

    def _bundle_path(self):
        """Find the model bundle inside the image.

        The image's own entrypoint hardcodes a path that is not there, so look for the
        bundle where it actually ships.
        """
        found = _sh(
            f"unshare -r apptainer exec {self.image} "
            "sh -c 'ls -d ${ERSILIA_PATH:-/opt/ersilia}/bundles/*/ 2>/dev/null | head -1'"
        ).stdout.strip().rstrip("/")
        if not found:
            raise RuntimeError(
                f"No model bundle inside {os.path.basename(self.image)}. "
                "This image may not be an ersilia-pack build."
            )
        return found

    def _free_port(self):
        """Stop whatever is serving, and wait for the port to come free.

        The apptainer wrapper exits at once, so the process holding the port is a
        run_uvicorn.py child; matching only ersilia_model_serve would kill nothing.
        """
        _sh("pkill -f 'ersilia_model_serve|run_uvicorn'")
        for _ in range(30):
            if not _sh(f"ss -lntH 'sport = :{self.port}'").stdout.strip():
                return
            time.sleep(1)
        raise RuntimeError(f"Port {self.port} is still in use.")

    def serve(self, limit=300):
        """Start the model server and wait until it answers."""
        if not os.path.exists(self.image):
            self.fetch()

        bundle = self._bundle_path()
        self._free_port()
        self.log.unlink(missing_ok=True)

        print(f"Starting {self.model_id}...")
        subprocess.Popen(
            f"unshare -r apptainer exec --bind /content:/content {self.image} "
            f"ersilia_model_serve --bundle_path {bundle} --port {self.port} "
            f"> {self.log} 2>&1",
            shell=True,
        )

        start = time.time()
        while time.time() - start < limit:
            text = self.log.read_text(errors="ignore") if self.log.exists() else ""
            if "CalledProcessError" in text or "Traceback (most recent call last)" in text:
                raise RuntimeError(
                    "The model did not start:\n" + "\n".join(text.splitlines()[-8:])
                )
            try:
                with urllib.request.urlopen(
                    f"http://127.0.0.1:{self.port}/healthz", timeout=5
                ) as response:
                    if response.status == 200:
                        self.serving = True
                        print(f"{self.model_id} is ready, after {time.time() - start:.0f}s.")
                        return self
            except (urllib.error.URLError, OSError):
                time.sleep(3)
        raise RuntimeError(f"No response from {self.model_id} after {limit}s. See {self.log}.")

    def run(self, input_list, batch=500):
        """Predict for a list of SMILES and return a DataFrame.

        The columns are the ones Ersilia itself produces: ``key``, ``input``, and then
        whatever the model returns.

        Serves the model first if it is not already running. Sends the molecules in
        batches, because each request costs about twenty seconds of fixed overhead.
        """
        if not self.serving:
            self.serve()

        # value != value catches NaN, which is what an empty cell in a CSV becomes.
        # Left as str() it would hash the literal "nan" and look like a real molecule.
        input_list = [
            "" if value is None or value != value else str(value).strip()
            for value in input_list
        ]
        blanks = input_list.count("")
        if blanks:
            print(f"Warning: {blanks} empty rows; they get a key of {UNPROCESSABLE_INPUT}.")
        rows = []
        for start in range(0, len(input_list), batch):
            chunk = input_list[start : start + batch]
            request = urllib.request.Request(
                f"http://127.0.0.1:{self.port}/run",
                data=json.dumps(chunk).encode(),
                headers={"Content-Type": "application/json"},
            )
            with urllib.request.urlopen(request, timeout=3600) as response:
                rows.extend(json.loads(response.read()))
            print(f"  {min(start + batch, len(input_list))} / {len(input_list)} molecules")

        frame = pd.DataFrame(rows)
        if len(frame) == len(input_list):
            # Ersilia's column order: key, then input, then the model's own columns.
            frame.insert(0, "input", input_list)
            frame.insert(0, "key", [compound_key(smiles) for smiles in input_list])
        else:
            print(f"Warning: sent {len(input_list)} molecules but got {len(frame)} rows back.")
        return frame

    def stop(self):
        """Stop the model server."""
        self._free_port()
        self.serving = False
        return self


def fetch_model(model_id, version="v1", port=8000):
    """Download a model, start it, and hand back something you can call run() on.

    Parameters
    ----------
    model_id : str
        Identifier from the Ersilia Model Hub, for example ``"eos42ez"``.
    version : str
        Version of the image, for example ``"v1"``.
    port : int
        Local port to serve the model on.

    Returns
    -------
    ErsiliaRunner
        A model that is already serving, ready for ``.run(input_list)``.

    Examples
    --------
    >>> model = fetch_model("eos42ez", "v1")
    >>> model.run(["CCO", "c1ccccc1"])
    """
    return ErsiliaRunner(model_id, version, port).fetch().serve()


_namespaces = 0
if os.path.exists("/proc/sys/user/max_user_namespaces"):
    _namespaces = int(open("/proc/sys/user/max_user_namespaces").read().strip() or 0)

if _namespaces == 0:
    print("FAILED: this runtime has user namespaces disabled, so containers cannot run here.")
elif shutil.which("apptainer"):
    print("Ready. ErsiliaRunner is defined.")
else:
    print("Installing Apptainer, about a minute...")
    _sh("add-apt-repository -y ppa:apptainer/ppa && apt-get update -qq && apt-get install -y apptainer")
    if shutil.which("apptainer"):
        print("Ready. ErsiliaRunner is defined.")
    else:
        print("FAILED: could not install Apptainer. Run this cell again.")


## 1. Fetch a model

Downloads the model's Apptainer image and starts it. Slow the first time, instant afterwards.


In [ ]:
model = fetch_model("eos42ez", "v1")


## 2. Run it

Pass a list of SMILES strings.


In [ ]:
smiles_list = ["CCO", "c1ccccc1", "C1CCCCC1"]
output_df = model.run(smiles_list)
output_df.head()


## 3. Save

Writes a CSV next to the notebook and downloads it to your computer.


In [ ]:
from google.colab import files

output_file = f"predictions_{model.model_id}_{model.version}.csv"
output_df.to_csv(output_file, index=False)
files.download(output_file)
